# M004 – Vintage Performance Mart Validation

## Objective

This notebook validates the Vintage Performance Mart built from the LendingClub feature store.

The mart aggregates loans by origination vintage using the issue_year_month field and provides a cohort-level view of credit quality, pricing, borrower leverage, utilization behaviour, and default performance.

Vintage analysis is one of the most important techniques used in credit risk management because it allows portfolio performance to be evaluated based on the period in which loans were originated.

The validation process confirms:

- Correct mart creation
- Correct vintage grain
- Loan count reconciliation
- Default rate reconciliation
- Vintage-level risk trends
- Cohort performance differences


In [10]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

DB_PATH = (
    Path.cwd()
    / "projects"
    / "P0_Data_Platform"
    / "datasets"
    / "lendingclub"
    / "data"
    / "warehouse"
    / "duckdb"
    / "lendingclub.duckdb"
)

print(DB_PATH)

conn = duckdb.connect(str(DB_PATH))

d:\project_lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb


## Mart Row Count Validation

The mart should contain one row per origination vintage.

A vintage represents all loans originated within a particular year-month combination.

In [11]:
conn.execute('''
select count(*) as mart_rows
from mart.vintage_performance
''').fetchdf()

,mart_rows
0,139


## Findings

The mart contains 139 vintages.

This confirms that the LendingClub portfolio spans a long origination history from 2007 through 2018 and that the mart successfully aggregated more than 2.2 million loans into monthly origination cohorts.

The resulting structure is appropriate for cohort analysis and portfolio performance monitoring.

## Grain Validation

The intended grain of this mart is:

**One row per issue_year_month**

The following validation confirms that each vintage appears only once.

In [12]:
conn.execute('''
select
    count(*) as rows,
    count(distinct issue_year_month) as unique_vintages
from mart.vintage_performance
''').fetchdf()

,rows,unique_vintages
0,139,139


## Findings

The mart contains 139 rows and 139 unique vintages.

This confirms that the aggregation logic is correct and that no duplicate vintage records exist.

The mart can therefore be safely used for cohort-level analysis without introducing double-counting errors.

## Loan Count Reconciliation

The total number of loans represented in the mart should reconcile exactly to the feature store population.

In [13]:
conn.execute('''
select
(
    select count(*)
    from feature.loan_features_v1
) as feature_rows,
(
    select sum(loan_count)
    from mart.vintage_performance
) as mart_loan_count
''').fetchdf()

,feature_rows,mart_loan_count
0,2260668,2260668.0


## Findings

The reconciliation is exact.

Feature Store Loans: 2,260,668

Mart Loans: 2,260,668

This confirms that all loans are represented within the mart and that no records were lost during aggregation.

## Schema Review

The schema should contain portfolio volume, borrower quality, pricing, utilization, leverage, and performance metrics required for vintage analysis.

In [14]:
conn.execute('''
describe mart.vintage_performance
''').fetchdf()

,column_name,column_type,null,key,default,extra
0,issue_year_month,VARCHAR,YES,None,None,None
1,loan_count,BIGINT,YES,None,None,None
2,total_loan_amount,DOUBLE,YES,None,None,None
3,avg_loan_amount,DOUBLE,YES,None,None,None
4,avg_fico,DOUBLE,YES,None,None,None
5,avg_dti,DOUBLE,YES,None,None,None
6,avg_interest_rate,DOUBLE,YES,None,None,None
7,avg_loan_to_income_ratio,DOUBLE,YES,None,None,None
8,avg_revolving_utilization_ratio,DOUBLE,YES,None,None,None
9,default_count,HUGEINT,YES,None,None,None


## Findings

The mart contains the expected metrics:

- Loan volume
- Exposure
- Average loan size
- Average FICO
- Average DTI
- Average interest rate
- Loan-to-income ratio
- Revolving utilization
- Default rate
- High utilization rate
- High DTI rate

The structure is suitable for portfolio monitoring and future underwriting trend analysis.

## Default Rate Reconciliation

The portfolio default rate reconstructed from the mart should match the feature-store default rate exactly.

In [15]:
conn.execute('''
select
(
    select round(avg(default_flag) * 100, 4)
    from feature.loan_features_v1
) as feature_default_rate,
(
    select round(sum(default_count) * 100.0 / sum(loan_count), 4)
    from mart.vintage_performance
) as mart_default_rate
''').fetchdf()

,feature_default_rate,mart_default_rate
0,12.8646,12.8646


## Findings

The portfolio default rate reconciles perfectly.

Feature Store Default Rate: 12.8646%

Mart Default Rate: 12.8646%

This confirms that default counts and loan counts were aggregated correctly and that the mart preserves portfolio-level risk metrics.

## Early Vintage Analysis

The following output displays the earliest LendingClub vintages available within the dataset.

In [16]:
conn.execute('''
select *
from mart.vintage_performance
order by issue_year_month
limit 20
''').fetchdf()

,issue_year_month,loan_count,total_loan_amount,avg_loan_amount,avg_fico,avg_dti,avg_interest_rate,avg_loan_to_income_ratio,avg_revolving_utilization_ratio,default_count,default_rate,high_utilization_rate,high_dti_rate
0,2007-06,24,91850.0,3827.083333,714.916667,8.690417,9.814583,0.134166,60.333333,3.0,12.50,0.00,0.0
1,2007-07,63,348325.0,5528.968254,702.079365,8.779683,11.158571,0.161717,40.762069,7.0,11.11,7.94,0.0
2,2007-08,74,515300.0,6963.513514,701.324324,9.761622,11.543514,0.169355,43.511765,20.0,27.03,13.51,0.0
3,2007-09,53,372950.0,7036.792453,686.245283,8.310943,12.463208,0.165644,48.403774,13.0,24.53,22.64,0.0
4,2007-10,105,753225.0,7173.571429,685.857143,10.693524,12.438476,0.177483,48.874286,34.0,32.38,19.05,0.0
5,2007-11,112,1008650.0,9005.803571,685.392857,11.537232,11.962321,0.194702,49.478571,34.0,30.36,21.43,0.0
6,2007-12,172,1887175.0,10971.947674,690.808140,12.315814,11.810523,0.207158,47.620349,47.0,27.33,20.93,0.0
7,2008-01,305,2926000.0,9593.442623,696.196721,12.941180,11.720393,0.186176,47.159672,84.0,27.54,20.00,0.0
8,2008-02,306,2959225.0,9670.669935,698.029412,13.233464,12.195915,0.182945,49.265686,61.0,19.93,19.61,0.0
9,2008-03,402,4150050.0,10323.507463,693.940299,12.889129,12.305771,0.186328,48.887688,89.0,22.14,14.43,0.0


## Findings

Several important patterns are visible within the earliest vintages.

### 1. Small Portfolio Size

Early vintages contain very small loan populations, often fewer than a few hundred loans.

### 2. Smaller Loan Amounts

Average loan sizes were generally below $10,000.

### 3. Lower Credit Quality

Several early vintages exhibit average FICO scores below 700.

### 4. Elevated Default Rates

Multiple early vintages experienced default rates above 20%, with some exceeding 30%.

These levels are materially higher than long-term portfolio averages and indicate that LendingClub's underwriting framework matured significantly over time.

## Highest Default Rate Vintage Analysis

The following output identifies the vintages that experienced the worst observed credit performance.

In [17]:
conn.execute('''
select *
from mart.vintage_performance
order by default_rate desc
limit 20
''').fetchdf()

,issue_year_month,loan_count,total_loan_amount,avg_loan_amount,avg_fico,avg_dti,avg_interest_rate,avg_loan_to_income_ratio,avg_revolving_utilization_ratio,default_count,default_rate,high_utilization_rate,high_dti_rate
0,2007-10,105,753225.0,7173.571429,685.857143,10.693524,12.438476,0.177483,48.874286,34.0,32.38,19.05,0.00
1,2007-11,112,1008650.0,9005.803571,685.392857,11.537232,11.962321,0.194702,49.478571,34.0,30.36,21.43,0.00
2,2008-01,305,2926000.0,9593.442623,696.196721,12.941180,11.720393,0.186176,47.159672,84.0,27.54,20.00,0.00
3,2007-12,172,1887175.0,10971.947674,690.808140,12.315814,11.810523,0.207158,47.620349,47.0,27.33,20.93,0.00
4,2007-08,74,515300.0,6963.513514,701.324324,9.761622,11.543514,0.169355,43.511765,20.0,27.03,13.51,0.00
5,2007-09,53,372950.0,7036.792453,686.245283,8.310943,12.463208,0.165644,48.403774,13.0,24.53,22.64,0.00
6,2008-04,259,2433875.0,9397.200772,696.227799,15.060927,12.381004,0.163201,55.716342,63.0,24.32,25.87,0.00
7,2008-03,402,4150050.0,10323.507463,693.940299,12.889129,12.305771,0.186328,48.887688,89.0,22.14,14.43,0.00
8,2008-05,115,628900.0,5468.695652,697.130435,14.509652,11.550957,0.109423,50.554783,25.0,21.74,20.87,0.00
9,2008-07,141,856475.0,6074.290780,692.460993,13.169645,11.905248,0.133768,49.666906,30.0,21.28,21.99,0.00


## Findings

The highest default-rate vintages are concentrated in the earliest years of LendingClub's operating history.

Examples include:

- 2007-10: 32.38%
- 2007-11: 30.36%
- 2008-01: 27.54%
- 2007-12: 27.33%
- 2007-08: 27.03%

Several characteristics are visible within these vintages:

- Lower average FICO scores
- Higher utilization levels
- Elevated borrower leverage
- Smaller portfolio sizes

### Business Interpretation

The results suggest that LendingClub's underwriting standards improved materially over time.


# Validation Conclusion

The Vintage Performance Mart successfully passed all validation controls.

### Controls Passed

✓ Mart created successfully

✓ Correct vintage grain

✓ Unique vintage records

✓ Loan count reconciliation

✓ Default rate reconciliation

✓ Cohort performance analysis

✓ Underwriting trend assessment

### Key Business Findings

The LendingClub portfolio contains 139 monthly origination vintages spanning 2007 through 2018.

The earliest vintages exhibit significantly higher default rates, lower borrower quality, and smaller portfolio sizes compared with later periods.

The results indicate that LendingClub's underwriting framework improved materially as the platform matured.

### Business Value

The Vintage Performance Mart provides the foundation for:

- Cohort monitoring
- Underwriting assessment
- Risk trend analysis
- Portfolio surveillance
- Future IFRS9 and portfolio analytics initiatives

### Validation Outcome

M004 – Vintage Performance Mart is complete and validated.


In [ ]:
conn.close()

: 